# 03 — Vector Databases, HNSW & Semantic Search

**Day 2 | Data Engineering & AI Bootcamp**

You have embeddings. Now you need to search them **fast** across millions of documents.
This notebook covers: ChromaDB indexing, HNSW algorithm, distance metrics,
semantic vs keyword search, and validating a 100+ document index.

**Stack:** ChromaDB (in-memory, zero config) + sentence-transformers + scikit-learn TF-IDF

In [ ]:
import sys
sys.path.insert(0, '../src')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time

from day2.vector_store import (
    distance_metrics_demo, HNSWConfig,
    ChromaVectorStore, KeywordSearch,
    generate_corpus, validate_index,
    semantic_vs_keyword_demo,
)

sns.set_theme(style='whitegrid')
print('Setup complete ✓')

## 1. Why Vector Databases?

Traditional databases answer: *'Find all rows where price < 100.'* — exact comparisons.
Vector databases answer: *'Find the 5 most similar embeddings to this query.'* — approximate similarity.
The key operation is **nearest-neighbour search** in high-dimensional space.

> **Real world:** When you ask Claude or ChatGPT a question about your company's documents,
> a vector database finds the relevant paragraphs in milliseconds from a million-page corpus.
> Databricks Vector Search, Pinecone, Weaviate, and pgvector all do this at production scale.

In [ ]:
vdb_comparison = {
    'ChromaDB':                 'Local / self-hosted. Zero config. Perfect for dev & prototyping.',
    'pgvector':                 'PostgreSQL extension. Add vector columns to your existing Postgres DB.',
    'Weaviate':                 'Self-hosted or cloud. Built-in BM25 hybrid search. GraphQL API.',
    'Pinecone':                 'Fully managed SaaS. Zero infra ops. Pay-per-vector.',
    'Databricks Vector Search': 'Native Delta Lake integration. Sync from Delta table → index automatically.',
    'Qdrant':                   'High-performance self-hosted. Excellent filtering and payload support.',
}

print('Vector Database Options:')
for name, desc in vdb_comparison.items():
    print(f'  {name:<30} {desc}')

print(f'\nToday: ChromaDB (in-memory) — no setup, no API keys, runs anywhere')

## 2. Distance Metrics: Cosine, Euclidean, Dot Product

Different distance metrics answer different questions:
- **Cosine similarity**: measures the **angle** between vectors. Ignores magnitude.
  Best for text embeddings. 'cat' and 'cats' point in the same direction.
- **Euclidean distance**: straight-line distance. Sensitive to vector magnitude.
  Better for spatial data (image patches, GPS coordinates).
- **Dot product**: cosine × magnitude. Rewards both direction AND length.
  Used when you want 'prominent' (high-norm) documents to score higher.

> **Rule:** Always L2-normalise your embeddings. Then cosine similarity = dot product.
> Use cosine (or dot product) for semantic search. Use Euclidean for geometric clustering.

In [ ]:
results = distance_metrics_demo()

metrics = ['cosine', 'euclidean', 'dot_product']
pairs   = list(results.keys())

print('Distance Metric Comparison:\n')
header = f'{"Pair":<35} {"Description":<30} {"Cosine":>8} {"Euclidean":>10} {"Dot Prod":>10}'
print(header)
print('─' * len(header))

for pair_key, data in results.items():
    print(f'{pair_key:<35} {data["description"]:<30} '
          f'{data["cosine"]:>8.4f} {data["euclidean"]:>10.4f} {data["dot_product"]:>10.4f}')

print()
print('Key insight: pair_a_d = same direction, 2x magnitude')
print('  Cosine    = 1.0000  (direction identical → maximum similarity)')
print('  Euclidean > 0       (distance > 0 because magnitude differs)')
print('  Dot prod  > cosine  (rewards the longer vector)')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

pair_labels = ['Similar\n(a,b)', 'Orthogonal\n(a,c)', 'Same dir\n(a,d)']
for ax, metric in zip(axes, ['cosine', 'euclidean', 'dot_product']):
    values = [results[p][metric] for p in pairs]
    colors = ['#2ecc71' if v > 0.5 else '#e74c3c' if metric == 'cosine' and v < 0.1
              else '#3498db' for v in values]
    bars = ax.bar(pair_labels, values, color=colors, alpha=0.85, edgecolor='white', width=0.5)
    for bar, v in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{v:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax.set_title(metric.replace('_', ' ').title(), fontweight='bold', fontsize=12)
    ax.set_ylim(min(0, min(values)) - 0.1, max(values) + 0.3)

plt.suptitle('Distance Metrics Compared on the Same Vector Pairs', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. HNSW — The Algorithm Behind Fast Vector Search

Brute-force nearest-neighbour: compare query to ALL N vectors → O(N) — too slow at scale.
**HNSW (Hierarchical Navigable Small World)**: graph-based approximate nearest-neighbour → O(log N).

How it works:
- **Layer 0:** contains ALL vectors with short-range connections.
- **Higher layers:** progressively fewer vectors with longer-range 'highway' connections.
- **Search:** enter at the top layer, greedily move towards the query, descend layer by layer.
- Result: approximate nearest neighbours in milliseconds even with 1 billion vectors.

> **Trade-off:** Higher `ef_search` → better recall but slower queries.
> In production, tune `ef_search` until recall meets your SLA.

In [ ]:
configs = [
    HNSWConfig(M=8,  ef_construction=100, ef_search=50),
    HNSWConfig(M=16, ef_construction=200, ef_search=100),
    HNSWConfig(M=32, ef_construction=400, ef_search=200),
]

print('HNSW Configuration Comparison:')
print(f'{"M":>4} {"ef_constr":>12} {"ef_search":>10} {"Est. Recall":>13} {"Memory":>10} {"Build":>10}')
print('─' * 65)
for cfg in configs:
    mem   = 'low'    if cfg.M <= 8  else ('medium' if cfg.M <= 16 else 'high')
    build = 'fast'   if cfg.ef_construction <= 100 else ('medium' if cfg.ef_construction <= 200 else 'slow')
    print(f'{cfg.M:>4} {cfg.ef_construction:>12} {cfg.ef_search:>10} '
          f'{cfg.recall_estimate():>13} {mem:>10} {build:>10}')

print('\nDefault recommendation: M=16, ef_construction=200, ef_search=100')
print('Adjust ef_search upward if recall < 95% on your benchmark set.')

## 4. Building an Index with 100+ Documents

Now we put it all together: generate 120 documents, embed them, and store in ChromaDB.
Each document has metadata (topic, priority) that enables filtered search.
This is the same workflow used in production RAG pipelines.

> **Real world:** At a company like Synechron, the document corpus might be 50,000 policy PDFs.
> The indexing job runs once (or nightly), and the vector store serves real-time queries.
> In Databricks, Delta Sync keeps the vector index automatically updated as Delta tables change.

In [ ]:
# Generate 120 documents across 6 topics
corpus_dicts = generate_corpus(120)
documents    = [d['text']  for d in corpus_dicts]
ids          = [d['doc_id'] for d in corpus_dicts]
metadatas    = [{'topic': d['topic'], 'priority': d['priority']} for d in corpus_dicts]

# Show distribution
topic_dist = pd.Series([d['topic'] for d in corpus_dicts]).value_counts()
print('Corpus distribution by topic:')
print(topic_dist.to_string())

# Build index
print(f'\nBuilding ChromaDB index...')
t0    = time.perf_counter()
store = ChromaVectorStore('main_corpus')
store.add_documents(documents, ids=ids, metadatas=metadatas)
elapsed = time.perf_counter() - t0
print(f'Indexed {store.count()} documents in {elapsed:.2f}s  ({store.count()/elapsed:.0f} docs/sec)')

## 5. Index Validation

After every bulk load, run validation checks before serving queries.
Key checks: correct count, no duplicate IDs, retrieval works correctly.
This prevents silent failures where the index is built but empty or corrupted.

> **Production pattern:** Add these as assertions in your ingestion DAG.
> If `count_ok` is False, halt the pipeline and alert. Never serve queries on a broken index.

In [ ]:
report = validate_index(store, expected_count=120)
print('Index Validation Report:')
for k, v in report.items():
    status = '✓' if v is True or (isinstance(v, (int, float)) and v > 0) else ''
    print(f'  {k:<25}: {v}  {status}')

assert report['count_ok'],      'Index count does not match expected!'
assert report['duplicate_ids'] == 0, 'Duplicate IDs found!'
assert report['retrieval_ok'],  'Retrieval returned empty results!'

print('\nAll index validation checks passed ✓')

## 6. Semantic Search Demo

Semantic search embeds the query and finds the nearest document vectors.
The query does NOT need to share words with the relevant documents — only meaning.
This is the core operation of every RAG system.

> **Example:** Query 'how do transformer models work' → returns documents about attention,
> BERT, and self-attention, even if none of them contain the exact word 'work'.

In [ ]:
queries = [
    ('how do transformers understand language',   None),
    ('measuring RAG pipeline quality',            None),
    ('fast approximate nearest neighbour index',  None),
    ('evaluation metrics for RAG only',           {'topic': 'rag'}),   # filtered
]

for query, where_filter in queries:
    hits = store.search(query, n_results=3, where=where_filter)
    filter_note = f' [filtered: {where_filter}]' if where_filter else ''
    print(f'\nQuery: "{query}"{filter_note}')
    for h in hits:
        print(f'  [{h["score"]:.4f}] [{h["metadata"].get("topic","")}] {h["document"][:80]}')

## 7. Semantic Search vs Keyword Search — Side by Side

Keyword search (TF-IDF/BM25) matches **exact tokens**. 'canine' ≠ 'dog'.
Semantic search matches **meaning**. 'canine' ≈ 'dog' in vector space.
Neither is always better — the right choice depends on your use case.

| | Keyword Search | Semantic Search |
|---|---|---|
| Exact product codes, IDs | ✓ Best | ✗ May miss |
| Paraphrased questions | ✗ Misses | ✓ Finds |
| Technical acronyms | ✓ Best | May miss |
| Multilingual queries | ✗ | ✓ With multilingual model |

> **Best practice: Hybrid search** — run both, combine scores.
> Weaviate's BM25 + vector fusion, Azure AI Search's Reciprocal Rank Fusion.

In [ ]:
# Queries where we expect semantic vs keyword to differ
test_queries = [
    'canine runs fast',                    # semantic wins: 'canine' -> 'dog' synonym mapping
    'HNSW approximate nearest neighbour',  # keyword wins: exact technical terminology match
    'understanding human language text',   # semantic wins: paraphrase of NLP concept
]

kw_search = KeywordSearch()
kw_search.index(documents)

print('=' * 80)
for query in test_queries:
    sem_hits = store.search(query, n_results=2)
    kw_hits  = kw_search.search(query, n=2)

    print(f'\nQuery: "{query}"')
    print(f'  Semantic [{sem_hits[0]["score"]:.4f}]: {sem_hits[0]["document"][:75]}')
    print(f'  Keyword  [{kw_hits[0]["score"]:.4f}]:  {kw_hits[0]["document"][:75]}')
    print(f'  Are they the same? {sem_hits[0]["document"][:30] == kw_hits[0]["document"][:30]}')